# POS Cash Balance – Feature Engineering

Step 1 of preprocessing for `POS_CASH_balance.csv`: monthly statements of
the applicant's previous point-of-sale (shop) and cash loans at Home Credit.
One row per loan (`SK_ID_PREV`) per month, with the number of instalments
and days past due (DPD = how many days a payment was late).

Needs `prev_loan_data_processed.parquet` from `Prevloans_data.ipynb` first,
for details of each previous loan.

**Sections**
1–2. Imports and settings
3. Load, keeping only applicants in the training data
4. Fill in missing instalment counts
5. One row per applicant per month
6. Lateness (DPD) features, all time and over recent periods
7. Early-repayment (prepayment) flag
8. Features from each loan's latest month
9. Months since the last late payment
10. Details from the previous-application data
11. Free memory
12. Combine everything, plus recent-vs-long-term ratios
13. Quality checks
14. Save
15. Free memory

**Output:** `data/Model-Training/Staging/posdata_processed.parquet` — one row per applicant.

Run by the `01_preprocessing` Airflow DAG (task `task_pos`, after
`task_prevloans`), or by hand.

## 1. Imports

In [32]:
import os
import pandas as pd
import numpy as np

## 2. Configuration & Paths

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker,
otherwise this project's folder on your PC.

In [33]:
# ── Project data folder ───────────────────────────────────────────────────────
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

# ── Inputs ────────────────────────────────────────────────────────────────
PATH_POS_BAL       = os.path.join(DATA_ROOT, "Input", "POS_CASH_balance.csv")
PATH_APP_TRAIN     = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_PREV_LOANS    = os.path.join(DATA_ROOT, "Model-Training", "Staging",
                                   "prev_loan_data_processed.parquet")   # from Prevloans_data.ipynb

# ── Output ────────────────────────────────────────────────────────────────
PATH_OUTPUT        = os.path.join(DATA_ROOT, "Model-Training", "Staging",
                                   "posdata_processed.parquet")

## 3. Load Data

Only applicants in `application_train.csv` are kept. `TARGET` is read with
the IDs but dropped straight after the join, so it can't leak into the features.

In [34]:
# Cleaned previous-loan details (used in section 10)
PrevLoans = pd.read_parquet(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

# Monthly POS statements, oldest month first within each loan
POSBal = pd.read_csv(PATH_POS_BAL)
POSBal = POSBal.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})
POSBal.sort_values(
    by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'],
    inplace=True,
    ignore_index=True,
)

# Training applicants
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

# Keep only training applicants' statements; drop TARGET straight away
POSBal = base.merge(POSBal, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

print(f"POSBal shape : {POSBal.shape}")
print(f"PrevLoans shape: {PrevLoans.shape}")

POSBal shape : (8543375, 8)
PrevLoans shape: (1413701, 40)


## 4. Fill in Missing Instalment Counts

`CNT_INSTALMENT` = the loan's total number of instalments, and
`CNT_INSTALMENT_FUTURE` = how many are still to pay. A missing month is
filled from the nearest month of the same loan (earlier months first), up
to 3 months away; longer gaps stay missing.

In [35]:
for col in ['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']:
    POSBal[col] = (
        POSBal
        .groupby('SK_ID_PREV')[col]
        .transform(lambda x: x.ffill(limit=3).bfill(limit=3))
    )

print("Remaining nulls after imputation:")
print(POSBal[['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']].isna().sum())

Remaining nulls after imputation:
CNT_INSTALMENT           1606
CNT_INSTALMENT_FUTURE    1633
dtype: int64


**Note:** worth looking further at prepayment-related features (see section 7).

## 5. One Row per Applicant per Month

When an applicant had several POS loans running in the same month, they're
combined into one row by taking the **highest** value of each column — so
each month shows the worst lateness across their loans.

In [36]:
POSBal_Mon = (
    POSBal
    .groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[
        ['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'SK_DPD', 'SK_DPD_DEF']
    ]
    .agg('max')
    .reset_index()
)

# Column names in capitals
POSBal_Mon.columns = POSBal_Mon.columns.str.upper()

print(f"POSBal_Mon shape: {POSBal_Mon.shape}")
POSBal_Mon.head(3)

POSBal_Mon shape: (7350685, 6)


,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,SK_DPD,SK_DPD_DEF
0,100002,-19,24.0,24.0,0,0
1,100002,-18,24.0,23.0,0,0
2,100002,-17,24.0,22.0,0,0


## 6. Lateness (DPD) Features

Per applicant, from the monthly table above:

- `POS_DAYS_IN_DPD` — months with a late payment (`SK_DPD` > 0; counts months despite the name)
- `POS_DAYS_IN_DPD_DEF` — months late beyond the small-debt tolerance (`SK_DPD_DEF` > 0)
- `POS_DPD_TO_DURATION_RATIO` — share of months that were late

Each is calculated over all history and over recent periods:

| Suffix | Period |
|--------|--------|
| *(none)* | All available history |
| `_LAST3Y` | Last 36 months (`MONTHS_BALANCE >= -36`) |
| `_LAST2Y` | Last 24 months (`MONTHS_BALANCE >= -24`) |
| `_LAST1Y` | Last 12 months (`MONTHS_BALANCE >= -12`) |
| `_LAST6M` | Last 6 months (`MONTHS_BALANCE >= -6`) |
| `_LAST3M` | Last 3 months (`MONTHS_BALANCE >= -3`) |

`MONTHS_BALANCE` counts months before the application (−1 = last month),
so `>= -12` means "in the last 12 months".

In [37]:
def build_dpd_features(df, suffix=''):
    """
    Lateness features per applicant (returned with SK_ID_CURR as the index).

    Parameters
    ----------
    df : monthly rows with SK_ID_CURR, SK_DPD, SK_DPD_DEF
    suffix : added to the end of each column name, e.g. '_LAST1Y'
    """
    return df.groupby('SK_ID_CURR').agg(
        **{
            f'POS_DAYS_IN_DPD{suffix}': ('SK_DPD', lambda x: (x > 0).sum()),                    # late months
            f'POS_DAYS_IN_DPD_DEF{suffix}': ('SK_DPD_DEF', lambda x: (x > 0).sum()),
            f'POS_DPD_TO_DURATION_RATIO{suffix}': ('SK_DPD', lambda x: (x > 0).sum() / len(x) if len(x) > 0 else 0),  # share late
        }
    )

pos_aggregated     = build_dpd_features(POSBal_Mon)
pos_aggregated_3Y  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -36], suffix='_LAST3Y'
)
pos_aggregated_2Y  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -24], suffix='_LAST2Y'
)
pos_aggregated_1Y  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -12], suffix='_LAST1Y'
)
pos_aggregated_6M  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -6], suffix='_LAST6M'
)
pos_aggregated_3M  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -3], suffix='_LAST3M'
)

print("All-time :", pos_aggregated.shape)
print("Last 36 M:", pos_aggregated_3Y.shape)
print("Last 24 M:", pos_aggregated_2Y.shape)
print("Last 12 M:", pos_aggregated_1Y.shape)
print("Last 6 M :", pos_aggregated_6M.shape)
print("Last 3 M :", pos_aggregated_3M.shape)

All-time : (289444, 3)
Last 36 M: (259354, 3)
Last 24 M: (245434, 3)
Last 12 M: (206897, 3)
Last 6 M : (166090, 3)
Last 3 M : (139631, 3)


## 7. Early-Repayment (Prepayment) Flag

A loan counts as **prepaid** when its total number of instalments
(`CNT_INSTALMENT`) in its first recorded month differs from its last
recorded month — the repayment schedule changed, e.g. it was paid off early.

`PREV_CREDIT_PREPAYMENT_FLAG` = 1 if any of the applicant's POS loans changed like this, else 0.

In [38]:
# Each loan's first and last recorded month
POSBal_first = POSBal.groupby(['SK_ID_CURR', 'SK_ID_PREV']).first().reset_index()
POSBal_last  = POSBal.groupby(['SK_ID_CURR', 'SK_ID_PREV']).last().reset_index()

# Compare the instalment count at the start and end of each loan
POSBal_CNT_INSTALMENT = (
    POSBal_first[['SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']]
    .merge(
        POSBal_last[['SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']],
        on=['SK_ID_CURR', 'SK_ID_PREV'],
        suffixes=('_FIRST', '_LAST'),
    )
)
POSBal_CNT_INSTALMENT['PREV_CREDIT_PREPAYMENT_FLAG'] = (
    POSBal_CNT_INSTALMENT['CNT_INSTALMENT_FIRST']
    != POSBal_CNT_INSTALMENT['CNT_INSTALMENT_LAST']
).astype(int)

# Per applicant: 1 if ANY of their loans was prepaid
POSBal_Prepay_flag = (
    POSBal_CNT_INSTALMENT
    .groupby('SK_ID_CURR')['PREV_CREDIT_PREPAYMENT_FLAG']
    .max()
    .reset_index()
)

print("Prepayment rate:",
      f"{POSBal_Prepay_flag['PREV_CREDIT_PREPAYMENT_FLAG'].mean():.2%}")

Prepayment rate: 52.27%


## 8. Features from Each Loan's Latest Month

From each loan's most recent recorded month (`POSBal_last`), summarised per
applicant: average and highest total/remaining instalments, how many loans
were still active, and the highest lateness in that month.

In [39]:
POSBal_Agg_last = POSBal_last.groupby('SK_ID_CURR').agg(
    POS_AVG_CNT_INSTALMENT_LAST_MONTH=('CNT_INSTALMENT', 'mean'),
    POS_AVG_CNT_INSTALMENT_FUTURE_LAST_MONTH=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_MAX_CNT_INSTALMENT_LAST_MONTH=('CNT_INSTALMENT', 'max'),
    POS_MAX_CNT_INSTALMENT_FUTURE_LAST_MONTH=('CNT_INSTALMENT_FUTURE', 'max'),
    POS_CNT_ACTIVE_PREV_CONTRACT_LAST_MONTH=('NAME_CONTRACT_STATUS', lambda x: (x == 'Active').sum()),
    POS_SK_DPD_LAST_MONTH=('SK_DPD', 'max'),
    POS_SK_DPD_DEF_LAST_MONTH=('SK_DPD_DEF', 'max'),
)

print(f"Last-month features shape: {POSBal_Agg_last.shape}")

Last-month features shape: (289444, 7)


## 9. Months Since the Last Late Payment

Takes only months with a late payment (`SK_DPD` > 0), then the most recent
one. `POS_MONTHS_SINCE_LAST_DPD_GT0` = how many months ago the applicant was
last late on any POS loan; blank if never late.

In [40]:
# Only months with a late payment, then the most recent of those
POSBal_latest_dpd = (
    POSBal[POSBal['SK_DPD'] > 0]          # months with a late payment
    .groupby('SK_ID_CURR')['MONTHS_BALANCE']
    .max()                                  # most recent (least negative) such month
    .reset_index()
    .rename(columns={'MONTHS_BALANCE': 'POS_MONTHS_SINCE_LAST_DPD_GT0'})
)
# MONTHS_BALANCE is negative; flip the sign to get "months ago"
POSBal_latest_dpd['POS_MONTHS_SINCE_LAST_DPD_GT0'] = (-POSBal_latest_dpd['POS_MONTHS_SINCE_LAST_DPD_GT0'])

print(f"Customers with at least one DPD event: {len(POSBal_latest_dpd):,}")
POSBal_latest_dpd['POS_MONTHS_SINCE_LAST_DPD_GT0'].describe()

Customers with at least one DPD event: 54,683


count    54683.000000
mean        42.139696
std         26.902085
min          1.000000
25%         18.000000
50%         41.000000
75%         65.000000
max         96.000000
Name: POS_MONTHS_SINCE_LAST_DPD_GT0, dtype: float64

## 10. Details from the Previous-Application Data

Adds each POS loan's details from `prev_loan_data_processed.parquet`, then
summarises per applicant: number of POS loans, how many were prepaid,
average amount granted ÷ amount asked for (`POS_AVG_CREDIT_TO_APPL`, used by
the current @prod model), average repayment ÷ amount, how many were in the
high/middle interest-rate groups, how many were insured, and how long ago
they were decided on average.

In [41]:
POSBal_PrevLoans = POSBal_CNT_INSTALMENT.merge(
    PrevLoans,
    on=['SK_ID_PREV', 'SK_ID_CURR'],
    how='left',
)

POSBal_Agg_PrevLoans = POSBal_PrevLoans.groupby('SK_ID_CURR').agg(
    POS_TOTAL_PREV_LOAN=('SK_ID_PREV', 'nunique'),
    POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG=('PREV_CREDIT_PREPAYMENT_FLAG', 'sum'),
    POS_AVG_CREDIT_TO_APPL=('CREDIT_TO_APPL', 'mean'),
    POS_ANNUITY_TO_CREDIT_RATIO=('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),
    POS_CNT_HIGH_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'high').sum()),
    POS_CNT_MIDDLE_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'middle').sum()),
    POS_CNT_INSURED_ON_APPROVAL=('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    POS_AVG_DAYS_DECISION=('DAYS_DECISION', 'mean'),
)

print(f"PrevLoan enrichment shape: {POSBal_Agg_PrevLoans.shape}")

PrevLoan enrichment shape: (289444, 8)


## 11. Free Memory

In [42]:
del PrevLoans, POSBal_Mon, POSBal_first, POSBal_last
del POSBal_CNT_INSTALMENT, POSBal_PrevLoans
import gc; gc.collect()
print("Intermediate DataFrames released.")

Intermediate DataFrames released.


## 12. Combine Everything

Starts from the all-time lateness features and joins on the rest, one row
per applicant. For each recent period a yes/no flag (`POS_IS_IN_LAST_3Y` …
`POS_IS_IN_LAST_3M`) says whether the applicant had any POS statement in it.

In [43]:
def flag_merge(left, right, on, flag_col):
    """Left-join `right` onto `left` and add `flag_col`: 1 if the row had a match in `right`, else 0."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


final_pos_aggregated = pos_aggregated.copy()

# Lateness over each recent period, plus a "had statements in this period" flag
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_3Y, 'SK_ID_CURR', 'POS_IS_IN_LAST_3Y'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_2Y, 'SK_ID_CURR', 'POS_IS_IN_LAST_2Y'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_1Y, 'SK_ID_CURR', 'POS_IS_IN_LAST_1Y'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_6M, 'SK_ID_CURR', 'POS_IS_IN_LAST_6M'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_3M, 'SK_ID_CURR', 'POS_IS_IN_LAST_3M'
)

# Latest-month features
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Agg_last, on='SK_ID_CURR', how='left'
)

# Months since the last late payment (blank if never late)
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_latest_dpd, on='SK_ID_CURR', how='left'
)

# Prepayment flag
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Prepay_flag, on='SK_ID_CURR', how='left'
)

# Previous-application details
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Agg_PrevLoans, on='SK_ID_CURR', how='left'
)

print(f"Final shape: {final_pos_aggregated.shape}")
final_pos_aggregated.head(3)

Final shape: (289444, 41)


,SK_ID_CURR,POS_DAYS_IN_DPD,POS_DAYS_IN_DPD_DEF,POS_DPD_TO_DURATION_RATIO,POS_DAYS_IN_DPD_LAST3Y,POS_DAYS_IN_DPD_DEF_LAST3Y,POS_DPD_TO_DURATION_RATIO_LAST3Y,POS_IS_IN_LAST_3Y,POS_DAYS_IN_DPD_LAST2Y,POS_DAYS_IN_DPD_DEF_LAST2Y,...,POS_MONTHS_SINCE_LAST_DPD_GT0,PREV_CREDIT_PREPAYMENT_FLAG,POS_TOTAL_PREV_LOAN,POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG,POS_AVG_CREDIT_TO_APPL,POS_ANNUITY_TO_CREDIT_RATIO,POS_CNT_HIGH_YIELD_GROUP,POS_CNT_MIDDLE_YIELD_GROUP,POS_CNT_INSURED_ON_APPROVAL,POS_AVG_DAYS_DECISION
0,100002,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,NaN,0,1,0,1.000000,0.051670,0,0,0.0,-606.0
1,100003,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,NaN,1,3,1,1.057664,0.126383,0,2,2.0,-1305.0
2,100004,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,NaN,1,1,1,0.828021,0.266450,0,1,0.0,-815.0


**Recent vs. long-term ratios:** each lateness feature's last-6-months value
÷ its last-3-years value. Above 1 means the applicant has been late more
often recently than over the longer term. Left blank when the 3-year value is 0.

In [44]:
def add_period_ratio_features(df, attributes, numerator_period='6M', denominator_period='3Y'):
    """
    For each attribute, add <attr>_LAST<numerator_period>_TO_<denominator_period>_RATIO
    = <attr>_LAST<numerator_period> ÷ <attr>_LAST<denominator_period>,
    left blank (NaN) when the denominator is 0.

    Both <attr>_LAST... columns must already exist in df (here they come
    from build_dpd_features in section 6).
    """
    for attr in attributes:
        numerator_col = f'{attr}_LAST{numerator_period}'
        denominator_col = f'{attr}_LAST{denominator_period}'
        ratio_col = f'{attr}_LAST{numerator_period}_TO_{denominator_period}_RATIO'
        df[ratio_col] = np.where(
            df[denominator_col] != 0,
            df[numerator_col] / df[denominator_col],
            np.nan,
        )
    return df

In [45]:
recent_vs_historical_attrs = [
    'POS_DAYS_IN_DPD',
    'POS_DAYS_IN_DPD_DEF',
    'POS_DPD_TO_DURATION_RATIO'
]

final_pos_aggregated = add_period_ratio_features(
    final_pos_aggregated, recent_vs_historical_attrs,
    numerator_period='6M', denominator_period='3Y',
)

## 13. Quality Checks

The share of missing values in the emptiest columns, and exactly one row per applicant.

In [46]:
# Null rates per feature
null_rates = final_pos_aggregated.isna().mean().sort_values(ascending=False)
print("Null rate (top 15 columns):")
print(null_rates.head(15).to_string())

# Duplicate customer IDs
n_dupes = final_pos_aggregated['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR rows found: {n_dupes}"
print(f"\nNo duplicate SK_ID_CURR rows. Final row count: {len(final_pos_aggregated):,}")

Null rate (top 15 columns):
POS_DAYS_IN_DPD_DEF_LAST6M_TO_3Y_RATIO          0.956444
POS_DAYS_IN_DPD_LAST6M_TO_3Y_RATIO              0.942918
POS_DPD_TO_DURATION_RATIO_LAST6M_TO_3Y_RATIO    0.942918
POS_MONTHS_SINCE_LAST_DPD_GT0                   0.811076
POS_DPD_TO_DURATION_RATIO_LAST3M                0.517589
POS_DAYS_IN_DPD_LAST3M                          0.517589
POS_DAYS_IN_DPD_DEF_LAST3M                      0.517589
POS_DAYS_IN_DPD_DEF_LAST6M                      0.426176
POS_DPD_TO_DURATION_RATIO_LAST6M                0.426176
POS_DAYS_IN_DPD_LAST6M                          0.426176
POS_DAYS_IN_DPD_LAST1Y                          0.285192
POS_DAYS_IN_DPD_DEF_LAST1Y                      0.285192
POS_DPD_TO_DURATION_RATIO_LAST1Y                0.285192
POS_DAYS_IN_DPD_DEF_LAST2Y                      0.152050
POS_DAYS_IN_DPD_LAST2Y                          0.152050

No duplicate SK_ID_CURR rows. Final row count: 289,444


## 14. Save

In [47]:
final_pos_aggregated.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\data\Model-Training\Staging\posdata_processed.parquet


## 15. Free Memory

In [48]:
del (
    pos_aggregated, pos_aggregated_1Y, pos_aggregated_2Y, pos_aggregated_3Y, pos_aggregated_6M, pos_aggregated_3M,
    POSBal_Agg_last, POSBal_latest_dpd, POSBal_Prepay_flag,
    POSBal_Agg_PrevLoans, base, POSBal,
)
import gc; gc.collect()
print("All DataFrames released.")

All DataFrames released.
